# upload Kaggle.json file

In [41]:
# configuring the path of Kaggle.json file
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

In [43]:
!kaggle datasets download -d kazanova/sentiment140

Dataset URL: https://www.kaggle.com/datasets/kazanova/sentiment140
License(s): other
sentiment140.zip: Skipping, found more recently modified local copy (use --force to force download)


In [44]:
# extracting the compressed dataset

from zipfile import ZipFile
dataset = '/content/sentiment140.zip'

with ZipFile(dataset, 'r') as zip:
  zip.extractall()
  print('the dataset is extracted')

the dataset is extracted


Importing the Dependencies

In [45]:
from os import access
import numpy as np
import pandas as pd
import re
from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

In [46]:
import nltk
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [47]:
# printing the stopwords in English
print(stopwords.words('english'))

['a', 'about', 'above', 'after', 'again', 'against', 'ain', 'all', 'am', 'an', 'and', 'any', 'are', 'aren', "aren't", 'as', 'at', 'be', 'because', 'been', 'before', 'being', 'below', 'between', 'both', 'but', 'by', 'can', 'couldn', "couldn't", 'd', 'did', 'didn', "didn't", 'do', 'does', 'doesn', "doesn't", 'doing', 'don', "don't", 'down', 'during', 'each', 'few', 'for', 'from', 'further', 'had', 'hadn', "hadn't", 'has', 'hasn', "hasn't", 'have', 'haven', "haven't", 'having', 'he', "he'd", "he'll", 'her', 'here', 'hers', 'herself', "he's", 'him', 'himself', 'his', 'how', 'i', "i'd", 'if', "i'll", "i'm", 'in', 'into', 'is', 'isn', "isn't", 'it', "it'd", "it'll", "it's", 'its', 'itself', "i've", 'just', 'll', 'm', 'ma', 'me', 'mightn', "mightn't", 'more', 'most', 'mustn', "mustn't", 'my', 'myself', 'needn', "needn't", 'no', 'nor', 'not', 'now', 'o', 'of', 'off', 'on', 'once', 'only', 'or', 'other', 'our', 'ours', 'ourselves', 'out', 'over', 'own', 're', 's', 'same', 'shan', "shan't", 'she

Data Processing

In [48]:
#  load the data and naming the columns name from csv file to pandas dataframe
columns_name = ["target", "id", "date",  "flag", "user", "text"]
twitter_data = pd.read_csv('/content/training.1600000.processed.noemoticon.csv', encoding='ISO-8859-1', names=columns_name)

In [49]:
negative = twitter_data[twitter_data['target'] == 0].sample(10000)
positive = twitter_data[twitter_data['target'] == 4].sample(10000)

twitter_data = pd.concat([negative, positive])

In [50]:
# checking the number of rows and columns
twitter_data.shape

(20000, 6)

In [51]:
# prinitng the forst 5 rows of the dataframe
twitter_data.head()

,target,id,date,flag,user,text
363764,0,2047943954,Fri Jun 05 14:25:09 PDT 2009,NO_QUERY,SupportMiley,"@mitchelmusso I wish I could go Mitchel, I rea..."
158518,0,1956624877,Thu May 28 22:17:45 PDT 2009,NO_QUERY,laceyllel,watching Greys anatomy finale... omg
486164,0,2181413829,Mon Jun 15 11:47:47 PDT 2009,NO_QUERY,rooppreet,accommodation issue over..shifting new place i...
731549,0,2263852419,Sun Jun 21 02:00:18 PDT 2009,NO_QUERY,rush,/sad. very.
39084,0,1573441866,Tue Apr 21 01:11:16 PDT 2009,NO_QUERY,DJ86,"Bonjour, ï¿½ava? Off for many meetings this mo..."


In [52]:
# counting the number if missing values in the dataset
twitter_data.isnull().sum()

,0
target,0
id,0
date,0
flag,0
user,0
text,0


Convert the target "4" to "1"

In [53]:
twitter_data.replace({'target':{4: 1}}, inplace=True)

In [54]:
# checking the distribution of target column
twitter_data['target'].value_counts()

,count
target,
0,10000
1,10000


0 -> Negative Tweet
1 -> Postive Tweet

# **Stemming**

Stemming is the process of reducing a word to its Root word

In [55]:
port_stem = PorterStemmer()

In [56]:
stop_words = set(stopwords.words('english'))

def stemming(content):
    stemmed_content = re.sub('[^a-zA-Z]', ' ', content)
    stemmed_content = stemmed_content.lower()
    words = stemmed_content.split()

    stemmed_words = [
        port_stem.stem(word)
        for word in words
        if word not in stop_words
    ]

    return ' '.join(stemmed_words)

In [57]:
twitter_data['stemmed_content'] = twitter_data['text'].apply(stemming)

In [58]:
twitter_data['stemmed_content']

,stemmed_content
363764,mitchelmusso wish could go mitchel realli
158518,watch grey anatomi final omg
486164,accommod issu shift new place weak miss platt ...
731549,sad
39084,bonjour ava mani meet morn think left late go ...
...,...
1373400,lol love save amp take miss sali itd nice u come
1416230,great day keith wait ian tomorr x
1287642,back civil gonna go walk town st lou chat litt...
913082,nikimoss take back colorado belong


In [59]:
twitter_data['target'].value_counts()

,count
target,
0,10000
1,10000


In [60]:
# separating the data and label
X = twitter_data['stemmed_content'].values
Y = twitter_data['target'].values

In [25]:
print(X)

['ritasitalianic yeah suppos go pool parti otday weather fail'
 'tomorrow last day quot see quot marii annaax'
 'babi dede ye look sooo funni lol let know' ... 'watch firefli'
 'deannacarlock new daughter love near imagin hope settl'
 'tiffmcmillan twitter problem night u today']


In [24]:
print(Y)

[0 0 0 ... 1 1 1]


Splitting the data to training data and test data

In [61]:
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, stratify=Y, random_state = 2)

In [62]:
print(X.shape, X_train.shape, X_test.shape)

(20000,) (16000,) (4000,)


In [28]:
print(Y.shape, Y_train.shape, Y_test.shape)

(20000,) (16000,) (4000,)


Training the machine learning model

Logistic Regression

In [63]:
# converting the textual data to numerical data
import pickle
vectorizer = TfidfVectorizer()

X_train = vectorizer.fit_transform(X_train)
X_test = vectorizer.transform(X_test)

model = LogisticRegression(max_iter=1000)
model.fit(X_train, Y_train)


pickle.dump(vectorizer, open('vectorizer.pkl', 'wb'))
pickle.dump(model, open('model.pkl', 'wb'))

In [30]:
print(X_train)

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 117960 stored elements and shape (16000, 21048)>
  Coords	Values
  (0, 11568)	0.5931460501072127
  (0, 17561)	0.3347818590436685
  (0, 11559)	0.551979850334878
  (0, 7938)	0.2670339635269456
  (0, 6737)	0.22031869266120233
  (0, 16491)	0.3340204348672573
  (1, 8007)	0.4975440335583051
  (1, 8224)	0.3190207360174023
  (1, 12359)	0.2416894704224845
  (1, 11886)	0.2209073886188293
  (1, 19921)	0.2725496244338628
  (1, 2421)	0.37587149492196475
  (1, 14169)	0.27831534359318855
  (1, 10372)	0.2725496244338628
  (1, 9921)	0.21892008738684215
  (1, 10803)	0.19937434214320193
  (1, 1262)	0.29745636739379416
  (2, 7938)	0.2411349260343602
  (2, 7447)	0.535618117752063
  (2, 6490)	0.27375276594001524
  (2, 6904)	0.4549542360497429
  (2, 18932)	0.2709815887686797
  (2, 9796)	0.3480999269236475
  (2, 802)	0.3285666419679789
  (2, 7304)	0.2654856433646949
  :	:
  (15996, 17914)	0.28997205652793473
  (15996, 4268)	0.2121848072989676
  (15

In [31]:
print(X_test)

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 25767 stored elements and shape (4000, 21048)>
  Coords	Values
  (0, 20436)	1.0
  (1, 6903)	0.215317520151387
  (1, 7010)	0.391491347008219
  (1, 10440)	0.4312498447427292
  (1, 10803)	0.48826435558034026
  (1, 18605)	0.34269527320468635
  (1, 20445)	0.3166877030719261
  (1, 20478)	0.3978231135301555
  (2, 138)	0.18995122760809555
  (2, 3014)	0.19353349619400007
  (2, 3361)	0.21533633421973408
  (2, 5903)	0.22230045360520714
  (2, 6063)	0.17691950328731498
  (2, 11116)	0.2565864863468302
  (2, 12753)	0.17491889223943655
  (2, 16027)	0.6114792076619533
  (2, 20103)	0.49713489630612856
  (2, 20445)	0.16565139796429518
  (2, 20499)	0.23334956999230955
  (3, 1513)	0.3077539954930372
  (3, 3492)	0.2081521400640937
  (3, 6267)	0.4364690028462948
  (3, 8036)	0.19576817633193422
  (3, 8220)	0.3743968602996861
  (3, 10172)	0.2239898579373055
  :	:
  (3996, 7418)	0.24932786801963
  (3996, 10574)	0.26300404723072107
  (3996, 12187)	0.2

Model Evaluation

Accuracy Score

In [64]:
# accuracy score on the training data
X_train_prediction = model.predict(X_train)
training_data_accuracy = accuracy_score(Y_train, X_train_prediction)

In [65]:
print('Accracy score on the training data :', training_data_accuracy)

Accracy score on the training data : 0.8538125


In [66]:
# accuracy score on the test data
X_test_prediction = model.predict(X_test)
test_data_accuracy = accuracy_score(Y_test, X_test_prediction)

In [67]:
print('Accracy score on the testing data :', test_data_accuracy)

Accracy score on the testing data : 0.74175


Model accuracy = 73.3%